# Multi-Quarter Approach A — Block-Diagonal Union

Train on the **union of K past quarter graphs** as one big disconnected graph,
test on the next quarter as in `gnn_bipartite_tertile_parquet.ipynb`.

Each historical quarter contributes its own subgraph — the same CIK at q-3 and
q-1 lives at *different* node indices, so this just augments the labeled set
(no temporal modelling, no cross-quarter leakage by construction).

Reads parquets from `~/13Fgnn/data/`, writes outputs to `~/13Fgnn/`.

## Setup

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("cuda build:", torch.version.cuda)

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv, GATConv

warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
OUT_DIR = Path(os.environ.get("FGNN_OUT_DIR", str(Path.home() / "13Fgnn"))).expanduser().resolve()
DATA_DIR = Path(os.environ.get("FGNN_DATA_DIR", str(OUT_DIR / "data"))).expanduser().resolve()
MODELS_DIR = OUT_DIR / "models"
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}. Set FGNN_DATA_DIR.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
torch.manual_seed(0)
np.random.seed(0)

print("OUT_DIR    :", OUT_DIR)
print("DATA_DIR   :", DATA_DIR)
print("MODELS_DIR :", MODELS_DIR)
print("device     :", DEVICE)

### Load parquets

In [ ]:
TGT_CHANGED_HOLDINGS = "changed_holdings"
TGT_STOCKS_RETURN = "stocks_return"
TGT_CIK_AUM = "cik_aum"
TGT_NORMALIZED_HOLDINGS = "normalized_holdings"
CUSIP_FIN_TABLE = "cusip_financial_data"


def _read_parquet_or_empty(name):
    p = DATA_DIR / f"{name}.parquet"
    if not p.exists():
        print(f"  [warn] missing parquet: {p.name} (returning empty df)")
        return pd.DataFrame()
    df = pd.read_parquet(p)
    print(f"  loaded {name:30s} {len(df):>10,} rows  {len(df.columns):>3} cols")
    return df


CHANGED_HOLDINGS = _read_parquet_or_empty(TGT_CHANGED_HOLDINGS)
STOCKS_RETURN = _read_parquet_or_empty(TGT_STOCKS_RETURN)
CIK_AUM = _read_parquet_or_empty(TGT_CIK_AUM)
NORM_HOLDINGS = _read_parquet_or_empty(TGT_NORMALIZED_HOLDINGS)
CUSIP_FIN = _read_parquet_or_empty(CUSIP_FIN_TABLE)

### Configuration

In [ ]:
STOCK_FEATURE_COLS = [
    "diluted_eps", "roe", "ev_ebitda", "pe_ratio", "price_to_sales",
    "price_to_book", "debt_to_equity", "dividend_yield", "fcf_per_share",
    "log_return",
]
EDGES_COLUMN_NAME = "change_in_weight"

# multi-quarter window: train uses K consecutive quarters ending at q-1; test = q.
K = 3

YEAR, QUARTER = 2019, 3
NUM_CLASSES = 3
HIDDEN_DIM = 64
NUM_LAYERS = 2
EPOCHS = 150
LR = 8e-4
DROPOUT = 0.5
GAT_HEADS = 4

## Data loaders (pandas / parquet)

In [ ]:
def next_year_quarter(year, quarter):
    return (year + 1, 1) if quarter == 4 else (year, quarter + 1)


def prev_year_quarter(year, quarter):
    return (year - 1, 4) if quarter == 1 else (year, quarter - 1)


def load_edges(year, quarter, edges_col_name):
    df = CHANGED_HOLDINGS
    if df.empty or edges_col_name not in df.columns:
        return pd.DataFrame(columns=["cik", "cusip", "w"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df[edges_col_name].notna()
    return (df.loc[mask, ["cik", "cusip", edges_col_name]]
             .rename(columns={edges_col_name: "w"})
             .reset_index(drop=True))


def load_returns(year, quarter):
    df = STOCKS_RETURN
    if df.empty:
        return pd.DataFrame(columns=["cusip", "log_return"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & df["log_return"].notna()
    return df.loc[mask, ["cusip", "log_return"]].reset_index(drop=True)


def load_aum(year, quarter):
    df = CIK_AUM
    if df.empty:
        return pd.DataFrame(columns=["cik", "aum"])
    mask = (df["year"] == year) & (df["quarter"] == quarter) & (df["total"] > 0)
    return (df.loc[mask, ["cik", "total"]]
             .rename(columns={"total": "aum"})
             .reset_index(drop=True))


def load_stock_features(year, quarter):
    fin = CUSIP_FIN
    if fin.empty:
        fin = pd.DataFrame(columns=["cusip"] + STOCK_FEATURE_COLS)
    else:
        fin = fin.loc[(fin["year"] == year) & (fin["quarter"] == quarter)].copy()
    rets = load_returns(year, quarter)
    df = fin.merge(rets, on="cusip", how="outer")
    for c in STOCK_FEATURE_COLS:
        if c not in df.columns:
            df[c] = 0.0
    return df[["cusip"] + STOCK_FEATURE_COLS]


def investor_profitability(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    h = NORM_HOLDINGS
    if h.empty:
        return pd.Series(dtype=float, name="profitability")
    h = h.loc[(h["year"] == year) & (h["quarter"] == quarter), ["cik", "cusip", "weight"]]
    r = load_returns(ny, nq)
    m = h.merge(r, on="cusip", how="inner")
    m["contrib"] = m["weight"] * m["log_return"]
    return m.groupby("cik")["contrib"].sum().rename("profitability")

## Tertile labeller & single-quarter graph builder

Identical to the single-quarter notebook — Approach A reuses `build_graph` as-is.

In [ ]:
def tertile_labels(values):
    s = values.astype(float)
    out = pd.Series(-1, index=s.index, dtype=np.int64)
    valid = s.dropna()
    if valid.empty:
        return out
    try:
        cats = pd.qcut(valid, q=3, labels=[0, 1, 2])
    except ValueError:
        q1, q2 = np.quantile(valid, [1 / 3, 2 / 3])
        cats = pd.cut(valid, bins=[-np.inf, q1, q2, np.inf], labels=[0, 1, 2], include_lowest=True)
    out.loc[valid.index] = cats.astype(np.int64)
    return out


def zscore(df, cols):
    out = df.copy()
    for c in cols:
        v = out[c].astype(float)
        v = v.replace([np.inf, -np.inf], np.nan).fillna(v.median() if v.notna().any() else 0.0)
        sd = v.std()
        out[c] = (v - v.mean()) / sd if sd > 0 else 0.0
    return out

In [ ]:
def _build_cik_features(edges, year, quarter):
    aum = load_aum(year, quarter)
    py, pq = prev_year_quarter(year, quarter)
    try:
        past_prof = investor_profitability(py, pq).reset_index()
    except Exception:
        past_prof = pd.DataFrame(columns=["cik", "profitability"])
    cik_nh = edges.groupby("cik").size().rename("n_holdings").reset_index()
    cik_df = (aum.merge(cik_nh, on="cik", how="outer")
                  .merge(past_prof, on="cik", how="left"))
    cik_df["aum"] = cik_df["aum"].fillna(
        cik_df["aum"].median() if cik_df["aum"].notna().any() else 0.0)
    cik_df["log_aum"] = np.log(cik_df["aum"].clip(lower=1.0))
    cik_df["n_holdings"] = cik_df["n_holdings"].fillna(0)
    cik_df["profitability"] = cik_df["profitability"].fillna(0.0)
    CIK_FEATS = ["log_aum", "n_holdings", "profitability"]
    return zscore(cik_df, CIK_FEATS), CIK_FEATS


def _build_stock_features(year, quarter):
    return zscore(load_stock_features(year, quarter), STOCK_FEATURE_COLS)


def _load_label_sources(year, quarter):
    ny, nq = next_year_quarter(year, quarter)
    r_next = load_returns(ny, nq).set_index("cusip")["log_return"]
    prof_next = investor_profitability(year, quarter)
    return r_next, prof_next


def _assemble_node_matrix(edges, cik_df, stock_df, CIK_FEATS):
    cik_ids = pd.Index(edges["cik"].unique())
    cusip_ids = pd.Index(edges["cusip"].unique())
    cik_df = cik_df.set_index("cik").reindex(cik_ids).fillna(0.0)
    stock_df = stock_df.set_index("cusip").reindex(cusip_ids).fillna(0.0)
    F_cik = cik_df[CIK_FEATS].to_numpy()
    F_stk = stock_df[STOCK_FEATURE_COLS].to_numpy()
    Fdim = F_cik.shape[1] + F_stk.shape[1]
    X = np.zeros((len(cik_ids) + len(cusip_ids), Fdim), dtype=np.float32)
    X[:len(cik_ids), :F_cik.shape[1]] = F_cik
    X[len(cik_ids):, F_cik.shape[1]:] = F_stk
    cik_pos = {c: i for i, c in enumerate(cik_ids)}
    cusip_pos = {c: i + len(cik_ids) for i, c in enumerate(cusip_ids)}
    return X, cik_ids, cusip_ids, cik_pos, cusip_pos


def _assemble_edges(edges, cik_pos, cusip_pos):
    src = edges["cik"].map(cik_pos).to_numpy()
    dst = edges["cusip"].map(cusip_pos).to_numpy()
    edge_index = np.stack(
        [np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    edge_weight = np.concatenate(
        [edges["w"].to_numpy(), edges["w"].to_numpy()]).astype(np.float32)
    return edge_index, edge_weight


def _assign_labels(num_nodes, cusip_pos, cik_pos, r_next, prof_next):
    y = np.full(num_nodes, -1, dtype=np.int64)
    if not r_next.empty:
        stk_lbl = tertile_labels(r_next)
        for cusip, idx in cusip_pos.items():
            v = stk_lbl.get(cusip, -1)
            if v >= 0:
                y[idx] = int(v)
    if not prof_next.empty:
        inv_lbl = tertile_labels(prof_next)
        for cik, idx in cik_pos.items():
            v = inv_lbl.get(cik, -1)
            if v >= 0:
                y[idx] = int(v)
    return y


def build_graph(year, quarter, edges_col_name):
    edges = load_edges(year, quarter, edges_col_name)
    if edges.empty:
        raise RuntimeError(f"no \u0394-edges for {year} Q{quarter}")
    cik_df, CIK_FEATS = _build_cik_features(edges, year, quarter)
    stock_df = _build_stock_features(year, quarter)
    r_next, prof_next = _load_label_sources(year, quarter)
    X, cik_ids, cusip_ids, cik_pos, cusip_pos = _assemble_node_matrix(
        edges, cik_df, stock_df, CIK_FEATS)
    edge_index, edge_weight = _assemble_edges(edges, cik_pos, cusip_pos)
    y = _assign_labels(X.shape[0], cusip_pos, cik_pos, r_next, prof_next)
    data = Data(
        x=torch.tensor(X),
        edge_index=torch.tensor(edge_index, dtype=torch.long),
        edge_weight=torch.tensor(edge_weight),
        y=torch.tensor(y),
    )
    data.is_cik = torch.zeros(X.shape[0], dtype=torch.bool)
    data.is_cik[:len(cik_ids)] = True
    data.has_label = data.y >= 0
    return data, {"n_cik": len(cik_ids), "n_cusip": len(cusip_ids)}

## Block-diagonal union

`union_quarters` stacks K independent quarter graphs into one disconnected
`Data` object. Node indices in subgraph `i` are shifted by the cumulative node
count of `[0..i-1]`, and edges are concatenated without any cross-subgraph
links — message passing within each subgraph is unchanged.

In [ ]:
def union_quarters(quarter_list, edges_col_name=EDGES_COLUMN_NAME):
    """Block-diagonal stack of build_graph(y, q) for each (y, q) in quarter_list."""
    parts = [build_graph(y, q, edges_col_name) for y, q in quarter_list]
    datas = [d for d, _ in parts]

    offsets, n = [], 0
    for d in datas:
        offsets.append(n)
        n += d.num_nodes

    x = torch.cat([d.x for d in datas], dim=0)
    y = torch.cat([d.y for d in datas], dim=0)
    is_cik = torch.cat([d.is_cik for d in datas], dim=0)
    has_label = torch.cat([d.has_label for d in datas], dim=0)
    edge_index = torch.cat(
        [d.edge_index + off for d, off in zip(datas, offsets)], dim=1)
    edge_weight = torch.cat([d.edge_weight for d in datas], dim=0)

    out = Data(x=x, edge_index=edge_index, edge_weight=edge_weight, y=y)
    out.is_cik = is_cik
    out.has_label = has_label
    out.quarters = list(quarter_list)
    return out


def past_K_quarters(target_year, target_quarter, K):
    """Returns [(y_K-back, q), ..., (y_1-back, q)] ending at target."""
    out = []
    y, q = target_year, target_quarter
    for _ in range(K):
        out.insert(0, (y, q))
        y, q = prev_year_quarter(y, q)
    return out

## Models, training, eval (unchanged)

In [ ]:
class WeightedSAGE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(SAGEConv(hidden_dim, hidden_dim))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)


class WeightedGAT(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=NUM_CLASSES, num_layers=2, heads=4, dropout=0.5):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout))
        for _ in range(num_layers - 2):
            self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=heads, dropout=dropout))
        self.convs.append(GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout))
        self.head = nn.Linear(hidden_dim, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return self.head(x)

In [ ]:
def train_one(model, data, train_mask, val_mask, epochs=EPOCHS, lr=LR, verbose=False):
    model = model.to(DEVICE)
    data = data.to(DEVICE)
    train_mask = train_mask.to(DEVICE)
    val_mask = val_mask.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

    best_val_acc = 0.0
    best_state = None
    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        logits = model(data.x, data.edge_index, data.edge_weight)
        loss = F.cross_entropy(logits[train_mask], data.y[train_mask])
        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index, data.edge_weight)
            pred = logits.argmax(dim=-1)
            val_acc = (
                (pred[val_mask] == data.y[val_mask]).float().mean().item()
                if val_mask.any() else 0.0
            )
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if verbose and ep % 25 == 0:
            print(f"  ep {ep:3d}  loss={loss.item():.4f}  val_acc={val_acc:.3f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def eval_subsets(model, data, mask):
    model.eval()
    with torch.no_grad():
        logits = model(
            data.x.to(DEVICE),
            data.edge_index.to(DEVICE),
            data.edge_weight.to(DEVICE),
        )
        pred = logits.argmax(dim=-1).cpu()
    y = data.y.cpu()
    out = {}
    for label, sel in [
        ("all", mask),
        ("stocks", mask & (~data.is_cik.cpu())),
        ("investors", mask & data.is_cik.cpu()),
    ]:
        if sel.sum() == 0:
            out[label] = {"n": 0}
            continue
        yt = y[sel].numpy()
        yp = pred[sel].numpy()
        out[label] = {
            "n": int(sel.sum()),
            "accuracy": accuracy_score(yt, yp),
            "macro_f1": f1_score(yt, yp, average="macro", labels=[0, 1, 2], zero_division=0),
        }
    return out

## Multi-quarter evaluation

Train on the union of `[q-K, …, q-1]`, test on `q`. Reuses the existing model
classes and `train_one` — no architectural change.

In [ ]:
def multiq_A_eval(year, quarter, K, model_cls, **kwargs):
    py, pq = prev_year_quarter(year, quarter)
    train_quarters = past_K_quarters(py, pq, K)
    train_data = union_quarters(train_quarters, EDGES_COLUMN_NAME)
    test_data, _ = build_graph(year, quarter, EDGES_COLUMN_NAME)

    Fdim = max(train_data.num_node_features, test_data.num_node_features)

    def pad(d):
        if d.num_node_features < Fdim:
            pad_cols = torch.zeros(d.num_nodes, Fdim - d.num_node_features)
            d.x = torch.cat([d.x, pad_cols], dim=1)
        return d

    train_data = pad(train_data)
    test_data = pad(test_data)

    model = model_cls(Fdim, HIDDEN_DIM, **kwargs)
    model = train_one(model, train_data, train_data.has_label, train_data.has_label, verbose=False)
    return model, eval_subsets(model, test_data, test_data.has_label), Fdim, train_data.num_nodes

In [ ]:
# sanity check on a single quarter
_, res_demo, fd_demo, n_train = multiq_A_eval(
    YEAR, QUARTER, K, WeightedSAGE,
    num_layers=NUM_LAYERS, dropout=DROPOUT,
)
print(f"K={K}  train nodes (union): {n_train:,}  feat dim: {fd_demo}")
pd.DataFrame(res_demo).T

## Per-quarter sweep

Iterates every target `(year, quarter)` where the K previous quarters AND the
next quarter (for test labels) all exist. Resumable: re-running picks up where
the CSV left off. Outputs land in `~/13Fgnn/models/`.

In [ ]:
import time
import pickle
import gc

RESULTS_CSV = MODELS_DIR / f"multiq_A_K{K}_results.csv"


def list_available_quarters(K):
    """Quarters where q-K..q-1, q, q+1 all exist (need q+1 for test labels)."""
    df = CHANGED_HOLDINGS
    sub = df.loc[df[EDGES_COLUMN_NAME].notna(), ["year", "quarter"]].drop_duplicates()
    avail = {(int(y), int(q)) for y, q in sub.itertuples(index=False)}
    out = []
    for y, q in sorted(avail):
        ny, nq = next_year_quarter(y, q)
        if (ny, nq) not in avail:
            continue
        # need K prior quarters before q
        py, pq = prev_year_quarter(y, q)
        ok = True
        for _ in range(K):
            if (py, pq) not in avail:
                ok = False
                break
            py, pq = prev_year_quarter(py, pq)
        if ok:
            out.append((y, q))
    return out


def load_done_set(csv_path):
    if not csv_path.exists():
        return set()
    df = pd.read_csv(csv_path)
    return set(zip(df["year"].astype(int), df["quarter"].astype(int)))


def append_row(row, csv_path):
    pd.DataFrame([row]).to_csv(
        csv_path, mode="a", header=not csv_path.exists(), index=False)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()


quarters = list_available_quarters(K)
print(f"Approach A  |  K={K}  |  {len(quarters)} target quarters: {quarters[0]} \u2026 {quarters[-1]}")
print(f"results -> {RESULTS_CSV}")

best = {"acc": -1.0, "model": None, "tag": None, "Fdim": None, "year": None, "quarter": None}
done = load_done_set(RESULTS_CSV)
remaining = [(y, q) for y, q in quarters if (y, q) not in done]
print(f"resume: {len(done)} done, {len(remaining)} remaining")

t_start = time.perf_counter()
for i, (y, q) in enumerate(remaining, 1):
    row = {"year": y, "quarter": q, "K": K}

    for tag, model_cls, kw in [
        ("sage", WeightedSAGE, dict(num_layers=NUM_LAYERS, dropout=DROPOUT)),
        ("gat",  WeightedGAT,  dict(num_layers=NUM_LAYERS, heads=GAT_HEADS, dropout=DROPOUT)),
    ]:
        try:
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            t0 = time.perf_counter()
            m, r, fd, n_train_nodes = multiq_A_eval(y, q, K, model_cls, **kw)
            row[f"{tag}_train_s"] = time.perf_counter() - t0
            row[f"{tag}_peak_gb"] = (torch.cuda.max_memory_allocated() / 1e9
                                       if torch.cuda.is_available() else 0.0)
            row[f"{tag}_train_nodes"] = n_train_nodes
            row[f"{tag}_acc"] = r["all"].get("accuracy")
            row[f"{tag}_f1"] = r["all"].get("macro_f1")
            row[f"{tag}_stocks_acc"] = r["stocks"].get("accuracy")
            row[f"{tag}_inv_acc"] = r["investors"].get("accuracy")
            if row[f"{tag}_acc"] is not None and row[f"{tag}_acc"] > best["acc"]:
                best.update({"acc": row[f"{tag}_acc"], "model": m.cpu(), "tag": tag,
                              "Fdim": fd, "year": y, "quarter": q})
        except Exception as e:
            print(f"  ! {tag.upper()} {y} Q{q} failed: {e.__class__.__name__}: {e}")
        finally:
            cleanup()

    append_row(row, RESULTS_CSV)
    elapsed = time.perf_counter() - t_start
    eta = elapsed / i * (len(remaining) - i)
    free_gb = (torch.cuda.mem_get_info()[0] / 1e9 if torch.cuda.is_available() else 0.0)
    print(f"  [{len(done) + i:2d}/{len(quarters)}] {y} Q{q}  "
          f"SAGE={row.get('sage_acc', float('nan')):.3f}  "
          f"GAT={row.get('gat_acc', float('nan')):.3f}  "
          f"free {free_gb:.2f}GB  ETA {eta/60:.1f}m")

if best["model"] is not None:
    best_path = MODELS_DIR / f"best_{best['tag']}_multiA_K{K}_{best['year']}Q{best['quarter']}.pkl"
    with open(best_path, "wb") as f:
        pickle.dump({
            "approach": "A_block_diagonal",
            "K": K,
            "tag": best["tag"],
            "year": best["year"],
            "quarter": best["quarter"],
            "Fdim": best["Fdim"],
            "test_acc": best["acc"],
            "state_dict": {k: v.cpu() for k, v in best["model"].state_dict().items()},
            "config": {
                "HIDDEN_DIM": HIDDEN_DIM,
                "NUM_LAYERS": NUM_LAYERS,
                "GAT_HEADS": GAT_HEADS,
                "DROPOUT": DROPOUT,
                "NUM_CLASSES": NUM_CLASSES,
                "EDGES_COLUMN_NAME": EDGES_COLUMN_NAME,
            },
        }, f)
    print(f"saved best -> {best_path}")

print(f"finished in {(time.perf_counter() - t_start)/60:.1f} min")
results_df = pd.read_csv(RESULTS_CSV) if RESULTS_CSV.exists() else pd.DataFrame()
results_df.tail()

## Aggregate summary

In [ ]:
df = pd.read_csv(RESULTS_CSV)
print(f"Approach A | K={K} | quarters: {len(df)} | random-chance acc = {1/NUM_CLASSES:.3f}\n")

print("accuracy mean +/- std:")
print(df[["sage_acc", "gat_acc"]].agg(["mean", "std"]).T.round(3))
print("\nmacro-F1 mean +/- std:")
print(df[["sage_f1", "gat_f1"]].agg(["mean", "std"]).T.round(3))
print("\ntrain time (s) mean:")
print(df[["sage_train_s", "gat_train_s"]].mean().round(2))

out_summary = OUT_DIR / f"multiq_A_K{K}_summary.csv"
df.describe().to_csv(out_summary)
print(f"\nsummary -> {out_summary}")